# 🦬 YAK RNA — An RNA Designer

Generate RNA sequences conditioned on:
- **Secondary structure** (dot-bracket notation)
- **Consensus sequence** (conservation patterns)
- **GO terms** (functional annotations)

[![GitHub](https://img.shields.io/github/stars/YousufAKhan/yakRNA?style=social)](https://github.com/YousufAKhan/yakRNA)

**Instructions:**
1. Run the **Setup** cell to install dependencies
2. Configure your **Input** parameters
3. Run **Generate** to design RNA sequences
4. Download results as FASTA

---

In [ ]:
#@title **Setup** { display-mode: "form" }
#@markdown Run this cell to install YAK RNA and download the model.
#@markdown This takes ~2 minutes on first run.

import os
import sys

# Clone repo if not already done
if not os.path.exists("yakRNA"):
    print("📥 Cloning YAK RNA repository...")
    
    # Try public clone first, fall back to authenticated if private
    result = os.system("git clone -q https://github.com/YousufAKhan/yakRNA.git 2>/dev/null")
    
    if result != 0:
        print("Repository is private. Please enter your GitHub Personal Access Token.")
        print("(Create one at: https://github.com/settings/tokens with 'repo' scope)")
        from getpass import getpass
        token = getpass("GitHub Token: ")
        os.system(f"git clone -q https://{token}@github.com/YousufAKhan/yakRNA.git")
        del token  # Clear token from memory
    
    print("✅ Repository cloned.")

os.chdir("yakRNA")
sys.path.append("inference")

# Display mascot
from IPython.display import display, Image
if os.path.exists("mascot.png"):
    display(Image("mascot.png", width=200))

# Install dependencies
print("📦 Installing dependencies...")
!pip install -q torch transformers accelerate biopython pyyaml h5py gdown

# Download model checkpoint from Google Drive
if not os.path.exists("checkpoint.pt"):
    print("🤖 Downloading model checkpoint...")
    import gdown
    gdown.download(id="19D0H9vDY5D_D6EUwhvDXl8cRBfH6idY3", output="checkpoint.pt", quiet=False)
    print("✅ Model checkpoint downloaded.")
else:
    print("✅ Model checkpoint found.")

print("\n✅ Setup complete!")

In [ ]:
#@title **GO Term Search** { display-mode: "form" }
#@markdown Search for GO terms by keyword. Results show GO IDs you can use in the Input cell.
#@markdown 
#@markdown The model supports **280 GO terms** (max 12 per sequence).

import json
import os

search_query = "ribosome"  #@param {type:"string"}
#@markdown Enter keywords like: `ribosome`, `translation`, `RNA binding`, `viral`, `splicing`

# Load GO term descriptions
go_file = "training_data/processed/vocabulary_analysis/go_term_descriptions.json"
if os.path.exists(go_file):
    with open(go_file) as f:
        go_terms_db = json.load(f)
    
    # Search
    query_lower = search_query.lower()
    matches = [(go_id, desc) for go_id, desc in go_terms_db.items() 
               if query_lower in desc.lower() or query_lower in go_id.lower()]
    
    if matches:
        print(f"🔍 Found {len(matches)} GO terms matching '{search_query}':\n")
        print(f"{'GO ID':<15} {'Description'}")
        print("-" * 60)
        for go_id, desc in sorted(matches, key=lambda x: x[1]):
            print(f"{go_id:<15} {desc}")
        print(f"\n💡 Copy GO IDs to use in the Input cell (comma-separated for multiple)")
    else:
        print(f"❌ No GO terms found matching '{search_query}'")
        print("   Try broader terms like: RNA, binding, regulation, viral, ribosome")
else:
    print("⚠️ GO term descriptions not found. Run Setup cell first.")

In [ ]:
#@title **Input** { display-mode: "form" }
#@markdown Configure your RNA generation parameters.
#@markdown
#@markdown **Limits:** Max sequence length: 636 | Max GO terms: 12

#@markdown ---
#@markdown ### Generation Settings
num_sequences = 5  #@param {type:"integer"}
temperature = 1.0  #@param {type:"slider", min:0.1, max:2.0, step:0.1}

#@markdown ---
#@markdown ### Conditioning Modalities
#@markdown Leave empty to skip a modality.

secondary_structure = ""  #@param {type:"string"}
#@markdown Example: `((((....))))` or `:::<<<___>>>:::`

consensus_sequence = ""  #@param {type:"string"}
#@markdown Example: `GAGUaaGGGG` (uppercase=conserved, lowercase=variable)

go_terms = ""  #@param {type:"string"}
#@markdown Example: `GO:0075523` or `GO:0003676,GO:0005515` (use GO Term Search to find IDs)

sequence_length = 80  #@param {type:"integer"}
#@markdown Only used if no structure/consensus provided (max: 636)

#@markdown ---
#@markdown ### Base-Pairing Constraints
constraint_set = "canonical"  #@param ["canonical", "strict", "canonical+sheared", "canonical+common", "permissive"]
#@markdown - `strict`: A:U, G:C only
#@markdown - `canonical`: + G:U wobble pairs (default)
#@markdown - `canonical+sheared`: + G:A pairs
#@markdown - `permissive`: most flexible

#@markdown ---
#@markdown ### Output
output_filename = "generated_sequences.fasta"  #@param {type:"string"}

# Validate inputs
if sequence_length > 636:
    print("⚠️ Warning: sequence_length exceeds max (636). Will be capped.")
    sequence_length = 636

if go_terms:
    go_count = len([g.strip() for g in go_terms.split(',') if g.strip()])
    if go_count > 12:
        print(f"⚠️ Warning: {go_count} GO terms provided, but max is 12. Only first 12 will be used.")

# Store parameters for generation
params = {
    'num_sequences': num_sequences,
    'temperature': temperature,
    'secondary_structure': secondary_structure,
    'consensus': consensus_sequence,
    'go_terms': go_terms,
    'length': sequence_length,
    'constraint_set': constraint_set,
    'output': output_filename
}

print("✅ Parameters configured!")
print(f"   Sequences: {num_sequences}")
print(f"   Temperature: {temperature}")
if secondary_structure:
    print(f"   Structure: {secondary_structure[:30]}{'...' if len(secondary_structure) > 30 else ''}")
if consensus_sequence:
    print(f"   Consensus: {consensus_sequence[:30]}{'...' if len(consensus_sequence) > 30 else ''}")
if go_terms:
    print(f"   GO Terms: {go_terms}")

In [ ]:
#@title **Generate** { display-mode: "form" }
#@markdown Run this cell to generate RNA sequences with your parameters.
#@markdown 
#@markdown If using secondary structure, constraint satisfaction stats will be displayed.

import subprocess
import os

# Build command
cmd = [
    "python", "inference/rna_sequence_generator.py",
    "--config", "configs/inference.yaml",
    "--checkpoint", "checkpoint.pt",
    "--num_sequences", str(params['num_sequences']),
    "--temperature", str(params['temperature']),
    "--fasta_output", params['output']
]

# Add optional parameters
if params['secondary_structure']:
    cmd.extend(["--secondary_structure", params['secondary_structure']])
    cmd.extend(["--constraint_set", params['constraint_set']])
elif params['length']:
    cmd.extend(["--length", str(params['length'])])

if params['consensus']:
    cmd.extend(["--consensus", params['consensus']])

if params['go_terms']:
    cmd.extend(["--go_terms", params['go_terms']])

print("🧬 Generating RNA sequences...\n")

# Run generation (output not captured so constraint stats are visible)
result = subprocess.run(cmd, text=True)

if result.returncode == 0 and os.path.exists(params['output']):
    print(f"\n{'='*50}")
    print(f"✅ Generated {params['num_sequences']} sequences")
    print(f"{'='*50}\n")
    
    # Display generated sequences
    with open(params['output'], 'r') as f:
        content = f.read()
        print(content)
else:
    print("\n❌ Generation failed. Check the error above.")

In [ ]:
#@title **Download Results** { display-mode: "form" }
#@markdown Download your generated sequences as a FASTA file.

from google.colab import files
import os

if os.path.exists(params['output']):
    print(f"📥 Downloading {params['output']}...")
    files.download(params['output'])
else:
    print("❌ No output file found. Run the Generate cell first.")

---
## Example Inputs

### Secondary Structure Notation
| Symbol | Meaning |
|--------|--------|
| `(` `)` | Base pair |
| `<` `>` | Base pair (alt) |
| `[` `]` | Base pair (alt) |
| `.` `:` | Unpaired |
| `_` | Unpaired (loop) |

### Consensus Sequence
- **Uppercase** (A, U, G, C): Conserved position
- **Lowercase** (a, u, g, c): Variable position  
- **Dots** (.): Any nucleotide

### GO Terms
Use Gene Ontology IDs like `GO:0075523` (viral RNA-directed RNA polymerase activity).

---

## Citation

```bibtex
@software{yakrna2025,
  author = {Khan, Yousuf},
  title = {YAK RNA: An RNA Designer},
  year = {2025},
  url = {https://github.com/YousufAKhan/yakRNA}
}
```